Since there is a retraining needed for poor accuracy values. So to hold up the FGP masks we need a interactive environment to retain and experiment with diffrent parameters.

In [1]:

import sys
import torch
from torch import nn
import copy
import random
import torch
from pathlib import Path

# Notebook is inside .../PruningNAS/PruningNAS, so add project root (.../PruningNAS)
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
	sys.path.insert(0, str(project_root))

# Install local package in editable mode (run once; safe to re-run)
%pip install -e ..


from PruningNAS.DataProcess.DataPreprocessing import get_dataloaders
from PruningNAS.Utills.EvaluatiorUtills import get_model_size, get_sparsity
from PruningNAS.Utills.PrunUtillCP import ChannelPrunner
from PruningNAS.Utills.PrunUtillFGP import FineGrainedPruner
from PruningNAS.Utills.TrainingModulesUtills import TrainingPrunned, evaluate
from PruningNAS.Utills.Utill import print_model
from PruningNAS.Utills.ViewerUtills import plot_accuracy, plot_loss  # Ensure you import your correct model architecture
seed=0
random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

ERROR: file:///D:/Sutanu_WorkSpace/PruningNas does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


Obtaining file:///D:/Sutanu_WorkSpace/PruningNas
Note: you may need to restart the kernel to use updated packages.
Device: cuda


d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Set Static Params here:

In [2]:
import os


current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")


Current working directory: d:\Sutanu_WorkSpace\PruningNAS\PruningNAS


In [3]:

# Initialize the model
basedir=''
path='./dataset/cifar10'
select_model='Densenet-121'
pruning_type='CP'
#model_path='./checkpoint/vgg_mrl_99.51375579833984.pth'
model_path=r'.\checkpoint\Densenet-121\Densenet-121_cifar_95.769997.pth'
# Load the saved state_dict correctly
model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

model.to(device)

sparsity_dict   = {

"init_conv.0": 0.0,

"blocks.0.block.0.conv1": 0.4,
"blocks.0.block.0.conv2": 0.0,
"blocks.0.block.1.conv1": 0.3,
"blocks.0.block.1.conv2": 0.0,
"blocks.0.block.2.conv1": 0.6,
"blocks.0.block.2.conv2": 0.0,
"blocks.0.block.3.conv1": 0.9,
"blocks.0.block.3.conv2": 0.0,
"blocks.0.block.4.conv1": 0.5,
"blocks.0.block.4.conv2": 0.0,
"blocks.0.block.5.conv1": 0.4,
"blocks.0.block.5.conv2": 0.0,

"blocks.2.block.0.conv1": 0.5,
"blocks.2.block.0.conv2": 0.0,
"blocks.2.block.1.conv1": 0.5,
"blocks.2.block.1.conv2": 0.0,
"blocks.2.block.2.conv1": 0.7,
"blocks.2.block.2.conv2": 0.0,
"blocks.2.block.3.conv1": 0.7,
"blocks.2.block.3.conv2": 0.0,
"blocks.2.block.4.conv1": 0.7,
"blocks.2.block.4.conv2": 0.0,
"blocks.2.block.5.conv1": 0.6,
"blocks.2.block.5.conv2": 0.0,
"blocks.2.block.6.conv1": 0.5,
"blocks.2.block.6.conv2": 0.0,
"blocks.2.block.7.conv1": 0.5,
"blocks.2.block.7.conv2": 0.0,
"blocks.2.block.8.conv1": 0.5,
"blocks.2.block.8.conv2": 0.0,
"blocks.2.block.9.conv1": 0.6,
"blocks.2.block.9.conv2": 0.0,
"blocks.2.block.10.conv1": 0.6,
"blocks.2.block.10.conv2": 0.0,
"blocks.2.block.11.conv1": 0.3,
"blocks.2.block.11.conv2": 0.0,

"blocks.4.block.0.conv1": 0.8,
"blocks.4.block.0.conv2": 0.0,
"blocks.4.block.1.conv1": 0.8,
"blocks.4.block.1.conv2": 0.0,
"blocks.4.block.2.conv1": 0.8,
"blocks.4.block.2.conv2": 0.0,
"blocks.4.block.3.conv1": 0.8,
"blocks.4.block.3.conv2": 0.0,
"blocks.4.block.4.conv1": 0.8,
"blocks.4.block.4.conv2": 0.0,
"blocks.4.block.5.conv1": 0.8,
"blocks.4.block.5.conv2": 0.0,
"blocks.4.block.6.conv1": 0.8,
"blocks.4.block.6.conv2": 0.0,
"blocks.4.block.7.conv1": 0.8,
"blocks.4.block.7.conv2": 0.0,
"blocks.4.block.8.conv1": 0.8,
"blocks.4.block.8.conv2": 0.0,
"blocks.4.block.9.conv1": 0.8,
"blocks.4.block.9.conv2": 0.0,
"blocks.4.block.10.conv1": 0.8,
"blocks.4.block.10.conv2": 0.0,
"blocks.4.block.11.conv1": 0.8,
"blocks.4.block.11.conv2": 0.0,
"blocks.4.block.12.conv1": 0.7,
"blocks.4.block.12.conv2": 0.0,
"blocks.4.block.13.conv1": 0.8,
"blocks.4.block.13.conv2": 0.0,
"blocks.4.block.14.conv1": 0.6,
"blocks.4.block.14.conv2": 0.0,
"blocks.4.block.15.conv1": 0.8,
"blocks.4.block.15.conv2": 0.0,
"blocks.4.block.16.conv1": 0.6,
"blocks.4.block.16.conv2": 0.0,
"blocks.4.block.17.conv1": 0.6,
"blocks.4.block.17.conv2": 0.0,
"blocks.4.block.18.conv1": 0.6,
"blocks.4.block.18.conv2": 0.0,
"blocks.4.block.19.conv1": 0.6,
"blocks.4.block.19.conv2": 0.0,
"blocks.4.block.20.conv1": 0.7,
"blocks.4.block.20.conv2": 0.0,
"blocks.4.block.21.conv1": 0.4,
"blocks.4.block.21.conv2": 0.0,
"blocks.4.block.22.conv1": 0.6,
"blocks.4.block.22.conv2": 0.0,
"blocks.4.block.23.conv1": 0.6,
"blocks.4.block.23.conv2": 0.0,

"blocks.6.block.0.conv1": 0.8,
"blocks.6.block.0.conv2": 0.0,
"blocks.6.block.1.conv1": 0.8,
"blocks.6.block.1.conv2": 0.0,
"blocks.6.block.2.conv1": 0.8,
"blocks.6.block.2.conv2": 0.0,
"blocks.6.block.3.conv1": 0.8,
"blocks.6.block.3.conv2": 0.0,
"blocks.6.block.4.conv1": 0.7,
"blocks.6.block.4.conv2": 0.0,
"blocks.6.block.5.conv1": 0.7,
"blocks.6.block.5.conv2": 0.0,
"blocks.6.block.6.conv1": 0.7,
"blocks.6.block.6.conv2": 0.0,
"blocks.6.block.7.conv1": 0.7,
"blocks.6.block.7.conv2": 0.0,
"blocks.6.block.8.conv1": 0.5,
"blocks.6.block.8.conv2": 0.0,
"blocks.6.block.9.conv1": 0.4,
"blocks.6.block.9.conv2": 0.0,
"blocks.6.block.10.conv1": 0.3,
"blocks.6.block.10.conv2": 0.0,
"blocks.6.block.11.conv1": 0.3,
"blocks.6.block.11.conv2": 0.0,
"blocks.6.block.12.conv1": 0.3,
"blocks.6.block.12.conv2": 0.0,
"blocks.6.block.13.conv1": 0.3,
"blocks.6.block.13.conv2": 0.0,
"blocks.6.block.14.conv1": 0.3,
"blocks.6.block.14.conv2": 0.0,
"blocks.6.block.15.conv1": 0.3,
"blocks.6.block.15.conv2": 0.0,
}

# sparsity_dict ={ 
# 'conv1':0.80,
# 'layer1.0.conv1':0.90,
# 'layer1.0.conv2':0.90,
# 'layer1.1.conv1':0.90,
# 'layer1.1.conv2':0.90,
# 'layer1.2.conv1':0.90,
# 'layer1.2.conv2':0.90,
# 'layer2.0.conv1':0.90,
# 'layer2.0.conv2':0.80,
# 'layer2.0.shortcut.0':0.80,
# 'layer2.1.conv1':0.90,
# 'layer2.1.conv2':0.90,
# 'layer2.2.conv1':0.90,
# 'layer2.2.conv2':0.90,
# 'layer2.3.conv1':0.90,
# 'layer2.3.conv2':0.90,
# 'layer3.0.conv1':0.90,
# 'layer3.0.conv2':0.80,
# 'layer3.0.shortcut.0':0.80,
# 'layer3.1.conv1':0.90,
# 'layer3.1.conv2':0.90,
# 'layer3.2.conv1':0.90,
# 'layer3.2.conv2':0.90,
# 'layer3.3.conv1':0.90,
# 'layer3.3.conv2':0.90,
# 'layer3.4.conv1':0.90,
# 'layer3.4.conv2':0.90,
# 'layer3.5.conv1':0.90,
# 'layer3.5.conv2':0.90,
# 'layer4.0.conv1':0.90,
# 'layer4.0.conv2':0.90,
# 'layer4.0.shortcut.0':0.90,
# 'layer4.1.conv1':0.90,
# 'layer4.1.conv2':0.90,
# 'layer4.2.conv1':0.90,
# 'layer4.2.conv2':0.90,
# 'fc':0.90,}


Define experimental params here:

In [4]:
num_finetune_epochs = 400
lr=0.1

In [5]:
train_dataloader,test_dataloader=get_dataloaders(path, batch_size=256 ) # Basemodel
dense_model_accuracy=evaluate(model,test_dataloader)
print('dense_model_accuracy:',dense_model_accuracy)


d:\Sutanu_WorkSpace\PruningNAS\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
                                                     

dense_model_accuracy: (95.7699966430664, 0.0)


In [6]:
pruned_model=copy.deepcopy(model)
if pruning_type=='FGP':
    isCallback=True
    pruner = FineGrainedPruner(pruned_model, sparsity_dict)
elif pruning_type=='CP':
    pruned_model=ChannelPrunner(pruned_model, sparsity_dict,select_model)
    pruner=None
    isCallback=False
else:
    print('pruning_type doesn\'t exists')
    exit

print_model(pruned_model)
print(f'The sparsity of each layer becomes')
for name, param in pruned_model.named_parameters():
    print(f'  {name}: {get_sparsity(param):.2f}')


init_conv.0.weight torch.Size([64, 3, 3, 3])
init_conv.1.weight torch.Size([64])
init_conv.1.bias torch.Size([64])
blocks.0.block.0.norm1.weight torch.Size([64])
blocks.0.block.0.norm1.bias torch.Size([64])
blocks.0.block.0.conv1.weight torch.Size([77, 64, 1, 1])
blocks.0.block.0.norm2.weight torch.Size([77])
blocks.0.block.0.norm2.bias torch.Size([77])
blocks.0.block.0.conv2.weight torch.Size([32, 77, 3, 3])
blocks.0.block.1.norm1.weight torch.Size([96])
blocks.0.block.1.norm1.bias torch.Size([96])
blocks.0.block.1.conv1.weight torch.Size([90, 96, 1, 1])
blocks.0.block.1.norm2.weight torch.Size([90])
blocks.0.block.1.norm2.bias torch.Size([90])
blocks.0.block.1.conv2.weight torch.Size([32, 90, 3, 3])
blocks.0.block.2.norm1.weight torch.Size([128])
blocks.0.block.2.norm1.bias torch.Size([128])
blocks.0.block.2.conv1.weight torch.Size([51, 128, 1, 1])
blocks.0.block.2.norm2.weight torch.Size([51])
blocks.0.block.2.norm2.bias torch.Size([51])
blocks.0.block.2.conv2.weight torch.Size([32,

In [ ]:
model_path=r'.\checkpoint\Densenet-121\CP\Densenet-121_cifar_CP_94.58999633789062.pth'
# Load the saved state_dict correctly
pruned_model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

pruned_model.to(device)

DenseNet(
  (init_conv): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (blocks): ModuleList(
    (0): DenseBlock(
      (block): Sequential(
        (0): DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu2): ReLU(inplace=True)
          (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (1): DenseLayer(
          (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(96, 12

In [9]:

dense_model_size = get_model_size(model, count_nonzero_only=True)
sparse_model_size = get_model_size(pruned_model, count_nonzero_only=True)

print(f"Sparse model has size={sparse_model_size:.2f} MiB = {sparse_model_size / dense_model_size * 100:.2f}% of dense model size")
sparse_model_accuracy,_ = evaluate(pruned_model, test_dataloader)
print(f"Sparse model has accuracy={sparse_model_accuracy:.2f}% before fintuning")
lr
optimizer = torch.optim.SGD(pruned_model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_finetune_epochs)
criterion = nn.CrossEntropyLoss()

Sparse model has size=12.33 MiB = 46.47% of dense model size


Sparse model has accuracy=94.49% before fintuning


In [ ]:
pruned_model_accuracy,best_pruned_model,accuracies,train_losses,test_losses=TrainingPrunned(pruned_model,train_dataloader,test_dataloader,criterion, optimizer, pruner,scheduler=scheduler,num_finetune_epochs=num_finetune_epochs,isCallback=isCallback)
print(pruned_model_accuracy)


In [12]:
print(pruned_model_accuracy)
basedir='.'

torch.save(best_pruned_model, f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}_{pruned_model_accuracy}.pth')

titel_append=f'of {pruning_type} based Pruned {select_model.title()} model'
save_path=f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}'

plot_accuracy(accuracies,titel_append=titel_append,save_path=save_path+'_acc.png' )
plot_loss(train_losses,test_losses,titel_append=titel_append,save_path=save_path+'_loss.png')

94.58999633789062
[66.75999450683594, 86.10999298095703, 87.41999816894531, 85.65999603271484, 88.29999542236328, 87.06999969482422, 87.15999603271484, 88.23999786376953, 85.73999786376953, 89.55000305175781, 89.08000183105469, 88.00999450683594, 90.0, 89.0, 89.86000061035156, 90.72000122070312, 90.11000061035156, 89.68000030517578, 90.45999908447266, 90.55000305175781, 90.4000015258789, 89.11000061035156, 89.43000030517578, 90.0999984741211, 89.86000061035156, 90.5999984741211, 89.61000061035156, 89.48999786376953, 90.31999969482422, 90.4800033569336, 90.76000213623047, 91.08000183105469, 91.18000030517578, 91.37999725341797, 90.83999633789062, 90.1500015258789, 91.78999328613281, 89.55000305175781, 90.54000091552734, 90.5999984741211, 90.27000427246094, 90.5, 91.0199966430664, 90.97999572753906, 89.93000030517578, 90.01000213623047, 90.09000396728516, 91.22999572753906, 91.06999969482422, 90.37999725341797, 90.95999908447266, 89.69000244140625, 90.41999816894531, 90.18000030517578, 9

In [10]:
best_pruned_model=pruned_model
best_pruned_model.cuda()

DenseNet(
  (init_conv): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (blocks): ModuleList(
    (0): DenseBlock(
      (block): Sequential(
        (0): DenseLayer(
          (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu2): ReLU(inplace=True)
          (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        )
        (1): DenseLayer(
          (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu1): ReLU(inplace=True)
          (conv1): Conv2d(96, 12

In [ ]:

num_finetune_epochs=200
optimizer = torch.optim.SGD(best_pruned_model.parameters(), lr=0.0001, momentum=0.9, weight_decay=1e-4)

pruned_model_accuracy,best_pruned_model,accuracies,train_losses,test_losses=TrainingPrunned(best_pruned_model,train_dataloader,test_dataloader,criterion, optimizer, pruner,scheduler=scheduler,num_finetune_epochs=num_finetune_epochs,isCallback=isCallback)


Finetuning Fine-grained Pruned Sparse Model


    Epoch 1 Test accuracy:66.76% / Best Accuracy: 66.76%, train loss: 0.2977, test loss: 1.2379


    Epoch 2 Test accuracy:86.11% / Best Accuracy: 86.11%, train loss: 0.3303, test loss: 0.4249


    Epoch 3 Test accuracy:87.42% / Best Accuracy: 87.42%, train loss: 0.2437, test loss: 0.4090


    Epoch 4 Test accuracy:85.66% / Best Accuracy: 87.42%, train loss: 0.1985, test loss: 0.4511


...

    Epoch 596 Test accuracy:94.49% / Best Accuracy: 94.59%, train loss: 0.0004, test loss: 0.2286


    Epoch 597 Test accuracy:94.43% / Best Accuracy: 94.59%, train loss: 0.0004, test loss: 0.2296


    Epoch 598 Test accuracy:94.48% / Best Accuracy: 94.59%, train loss: 0.0004, test loss: 0.2276


    Epoch 599 Test accuracy:94.52% / Best Accuracy: 94.59%, train loss: 0.0004, test loss: 0.2271


    Epoch 600 Test accuracy:94.51% / Best Accuracy: 94.59%, train loss: 0.0004, test loss: 0.2286
